In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()


# 01 - EDA
First: the REAL column names (they were not known at design time). The full CSV is never loaded in pandas: we peek at 3 rows and then query the parquet with DuckDB.

In [2]:
raw = pd.read_csv(C.RAW_FLIGHTS_CSV, nrows=3)
print(list(raw.columns))
raw

['FlightDate', 'Day_Of_Week', 'Airline', 'Tail_Number', 'Dep_Airport', 'Dep_CityName', 'DepTime_label', 'Dep_Delay', 'Dep_Delay_Tag', 'Dep_Delay_Type', 'Arr_Airport', 'Arr_CityName', 'Arr_Delay', 'Arr_Delay_Type', 'Flight_Duration', 'Distance_type', 'Delay_Carrier', 'Delay_Weather', 'Delay_NAS', 'Delay_Security', 'Delay_LastAircraft', 'Manufacturer', 'Model', 'Aicraft_age']


,FlightDate,Day_Of_Week,Airline,Tail_Number,Dep_Airport,Dep_CityName,DepTime_label,Dep_Delay,Dep_Delay_Tag,Dep_Delay_Type,...,Flight_Duration,Distance_type,Delay_Carrier,Delay_Weather,Delay_NAS,Delay_Security,Delay_LastAircraft,Manufacturer,Model,Aicraft_age
0,2023-01-02,1,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-3,0,Low <5min,...,56,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,16
1,2023-01-03,2,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,62,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,16
2,2023-01-04,3,Endeavor Air,N331PQ,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,49,Short Haul >1500Mi,0,0,0,0,0,CANADAIR REGIONAL JET,CRJ,10


In [3]:
print(q("describe select * from read_parquet('{G}')")[["column_name","column_type"]].to_string(index=False))

           column_name column_type
           flight_date        DATE
               airline     VARCHAR
           tail_number     VARCHAR
                origin     VARCHAR
                  dest     VARCHAR
        dep_time_label     VARCHAR
             arr_delay      DOUBLE
             dep_delay      DOUBLE
       flight_duration      DOUBLE
         delay_carrier      DOUBLE
         delay_weather      DOUBLE
             delay_nas      DOUBLE
        delay_security      DOUBLE
   delay_last_aircraft      DOUBLE
          is_cancelled     TINYINT
           is_diverted     TINYINT
          manufacturer     VARCHAR
                 model     VARCHAR
          aircraft_age      DOUBLE
                 month     TINYINT
           day_of_week     TINYINT
        distance_miles       FLOAT
origin_weather_missing     TINYINT
  dest_weather_missing     TINYINT
                o_tavg       FLOAT
                o_tmin       FLOAT
                o_tmax       FLOAT
                o_pr

In [4]:
tot = q("""select count(*) flights, sum(is_cancelled) cancelled,
  round(avg((arr_delay>=15)::int) filter (where is_cancelled=0),4) delay_rate,
  round(avg(is_cancelled),4) cancel_rate, min(flight_date) first_day, max(flight_date) last_day from read_parquet('{G}')""")
tot

,flights,cancelled,delay_rate,cancel_rate,first_day,last_day
0,6830352,86991.0,0.2056,0.0127,2023-01-01,2023-12-31


## Delay rate by airline, month, weekday, time of day

In [5]:
d = q("""select airline, count(*) n, avg((arr_delay>=15)::int) rate from read_parquet('{G}')
 where is_cancelled=0 group by 1 order by rate desc""")
fig, ax = plt.subplots(figsize=(8, 5)); sns.barplot(data=d, y="airline", x="rate", color="#3b82f6", ax=ax)
ax.set_xlabel("share of flights arriving >=15 min late"); ax.set_ylabel(""); save("eda_delay_by_airline")

In [6]:
m = q("""select month, avg((arr_delay>=15)::int) filter (where is_cancelled=0) delay_rate, avg(is_cancelled) cancel_rate
 from read_parquet('{G}') group by 1 order by 1""")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(m.month, m.delay_rate, marker="o"); ax[0].set_title("delay rate by month"); ax[0].axvspan(9.5, 12.5, alpha=.12, color="orange")
ax[1].plot(m.month, m.cancel_rate, marker="o", color="#dc2626"); ax[1].set_title("cancel rate by month"); ax[1].axvspan(9.5, 12.5, alpha=.12, color="orange")
save("eda_delay_by_month")   # shaded = test period (Oct-Dec): much lower base rate

In [7]:
d = q("""select day_of_week dow, dep_time_label, avg((arr_delay>=15)::int) rate from read_parquet('{G}')
 where is_cancelled=0 group by 1,2""")
piv = d.pivot(index="dow", columns="dep_time_label", values="rate")[["Night","Morning","Afternoon","Evening"]]
fig, ax = plt.subplots(figsize=(6, 4)); sns.heatmap(piv, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax); ax.set_title("delay rate: weekday (0=Sun) x time of day")
save("eda_delay_by_day_time")

## Weather and delay causes

In [8]:
w = q("""select case when o_prcp<0.1 then '0 dry' when o_prcp<5 then '1 light' when o_prcp<15 then '2 moderate' else '3 heavy' end prcp,
 count(*) n, avg((arr_delay>=15)::int) rate from read_parquet('{G}') where is_cancelled=0 group by 1 order by 1""")
fig, ax = plt.subplots(figsize=(6, 3.8)); sns.barplot(data=w, x="prcp", y="rate", color="#0ea5e9", ax=ax); ax.set_title("delay rate by origin precipitation band")
save("eda_delay_by_weather"); w

,prcp,n,rate
0,0 dry,4748697,0.180764
1,1 light,1173480,0.236539
2,2 moderate,479004,0.285459
3,3 heavy,342180,0.333111


In [9]:
c = q("""select sum(delay_carrier) Carrier, sum(delay_weather) Weather, sum(delay_nas) NAS, sum(delay_security) Security,
 sum(delay_last_aircraft) "Late aircraft" from read_parquet('{G}') where is_cancelled=0 and arr_delay>=15""").T
c.columns = ["minutes"]; c["share"] = c.minutes / c.minutes.sum()
fig, ax = plt.subplots(figsize=(6, 4)); ax.pie(c.minutes, labels=c.index, autopct="%1.0f%%", startangle=90); ax.set_title("delay minutes by cause (EDA only, never a feature)")
save("eda_delay_causes"); c

,minutes,share
Carrier,34831423.0,0.364116
Weather,5009341.0,0.052366
NAS,17309961.0,0.180953
Security,206677.0,0.002161
Late aircraft,38302747.0,0.400404


In [10]:
a = q("""select origin, count(*) n, avg((arr_delay>=15)::int) filter (where is_cancelled=0) rate from read_parquet('{G}')
 group by 1 having count(*)>40000 order by rate desc limit 15""")
fig, ax = plt.subplots(figsize=(7, 5)); sns.barplot(data=a, y="origin", x="rate", color="#f59e0b", ax=ax); ax.set_title("15 most delay-prone busy origins")
save("eda_delay_by_airport")

## Findings
- Base delay rate is about 20% overall but about 22% Jan-Sep vs about 15% Oct-Dec: the time split will show a shift.
- Cancellations are rare (about 1.3%) and seasonal (winter storms, summer convection).
- Dep_Delay and the Delay_* causes explain arrival delay almost perfectly, so they are leakage and are excluded from features.